# 01 · 공식 `ollama` 파이썬 클라이언트

`ollama.chat()`로 `/api/chat`을 부른다. 비스트리밍 → 스트리밍 → tok/s → 멀티턴 순서로 본다.

선행: Ollama가 켜져 있고 `qwen3:8b`가 받아져 있어야 한다(00-01).

In [1]:
import time
import ollama

MODEL = "qwen3:8b"
OPTIONS = {"num_ctx": 4096, "temperature": 0.7}   # 세 노트북이 같은 값을 쓴다
THINK = False   # qwen3 사고 과정(<think>) 끄기 — 화면 처리는 Phase 3 전에 정한다
MESSAGES = [
    {"role": "system", "content": "너는 간결하게 한국어로 답하는 도우미다."},
    {"role": "user", "content": "스트리밍 응답이 왜 체감 속도를 높이는지 세 문장으로 설명해줘."},
]

## 1. 비스트리밍
답변 전체가 한 번에 돌아온다. 응답 객체에 `eval_count`(생성 토큰 수)와 `eval_duration`(나노초)이 들어 있다.

In [2]:
r = ollama.chat(model=MODEL, messages=MESSAGES, options=OPTIONS, think=THINK)
print(r.message.content)
print('eval_count =', r.eval_count, '| eval_duration(ns) =', r.eval_duration)

스트리밍 응답은 데이터를 실시간으로 전송해 사용자가 즉시 결과를 받을 수 있게 해요.  
중간에 데이터를 저장하거나 기다리는 시간이 줄어들기 때문에 응답이 빨라져요.  
이로 인해 사용자 경험과 시스템 효율성이 동시에 향상돼요.
eval_count = 82 | eval_duration(ns) = 1850907000


## 2. 스트리밍 + tok/s
`stream=True`면 청크가 하나씩 온다. 마지막 청크(`done=True`)에 통계가 실린다.

- 청크 수가 1보다 훨씬 크면 조금씩 흘러온 것이다. 청크 하나가 정확히 토큰 하나는 아니므로(청크 수 ≤ 생성 토큰 수) 청크 수와 토큰 수는 다를 수 있다.
- tok/s = `eval_count` / (`eval_duration` / 1e9)  — Ollama가 잰 생성 구간 값이다.

> 참고: 「첫 토큰까지」 시간은 모델이 메모리에 없으면 적재 시간이 포함된다. 이미 올라가 있으면 0.1초 안팎이다.

In [3]:
t0 = time.perf_counter()
first = None
chunks = 0
last = None
for chunk in ollama.chat(model=MODEL, messages=MESSAGES, options=OPTIONS, think=THINK, stream=True):
    piece = chunk.message.content
    if piece:
        if first is None:
            first = time.perf_counter() - t0
        chunks += 1
        print(piece, end="", flush=True)
    last = chunk
print()
tps = last.eval_count / (last.eval_duration / 1e9)
print(f"청크 {chunks}개 | 첫 토큰까지 {first:.2f}s | 생성 {last.eval_count}토큰 | {tps:.1f} tok/s")

스트

리

밍

 응

답

...

아

진

다

.


청크 76개 | 첫 토큰까지 0.05s | 생성 90토큰 | 44.9 tok/s


## 3. 멀티턴 — 메시지 배열
Ollama는 대화를 기억하지 않는다. 앞선 `user`·`assistant` 메시지를 배열에 계속 붙여 매번 보낸다.

In [4]:
history = list(MESSAGES)
reply = ""
for chunk in ollama.chat(model=MODEL, messages=history, options=OPTIONS, think=THINK, stream=True):
    reply += chunk.message.content or ""
history.append({"role": "assistant", "content": reply})
history.append({"role": "user", "content": "방금 답을 한 문장으로 줄여줘."})

for chunk in ollama.chat(model=MODEL, messages=history, options=OPTIONS, think=THINK, stream=True):
    print(chunk.message.content or "", end="", flush=True)
print()
print("메시지 수:", len(history))

스트

리

밍

 응

답

...

 수

 있도록

 한다

.


메시지 수: 4
